# INST326 — Week 10 Exercises: Data Models, Properties, and Special Methods (Library Management)

**Focus (Week 10 only):** Python data model & class design refinements — `@property` (get/set/delete) with validation, dataclass options (equality, ordering, `field` options, `__post_init__`), special methods (`__str__`, `__repr__`, `__len__`, `__iter__`, `__contains__`, `__eq__`, ordering), and lightweight container patterns. Optional: `functools.total_ordering` and `dataclasses.replace`.

**Out of scope (Week 11+):** multiple inheritance/mixins, advanced design patterns, descriptors beyond `@property`, metaclasses, concurrency, ORMs, complex type-system features, network I/O, or frameworks.


### Starter Scaffold (Week-10-safe)

Below is minimal starter code from prior weeks, lightly adapted for Week 10. You may extend it while staying within the Week 10 scope.


In [205]:
from __future__ import annotations
from dataclasses import dataclass, field, replace
from datetime import datetime, timedelta
from typing import Dict, Iterable, Iterator, Optional, List
from functools import total_ordering

# --- Exceptions (basic) ---
class LibraryError(Exception): ...
class DuplicateBookError(LibraryError): ...
class OverdueLoanError(LibraryError): ...

@total_ordering
@dataclass(order=False)
class Book:
    isbn: str
    title: str
    _copies: int = field(default=1, repr=False, compare=False)

    # Comparable by title then ISBN (Week 10: ordering/special methods)
    def __lt__(self, other: "Book") -> bool:
        if not isinstance(other, Book):
            return NotImplemented
        return (self.title, self.isbn) < (other.title, other.isbn)

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Book):
            return NotImplemented
        return self.isbn == other.isbn  # equality by identity key

    def __repr__(self) -> str:
        return f"Book(isbn={self.isbn!r}, title={self.title!r}, copies={self._copies})"

    def __str__(self) -> str:
        return f"{self.title} [{self.isbn}]"

    # Week 10: properties for validation
    @property
    def copies(self) -> int:
        return self._copies

    @copies.setter
    def copies(self, value: int) -> None:
        if not isinstance(value, int):
            raise TypeError("copies must be int")
        if value < 0:
            raise ValueError("copies must be non-negative")
        self._copies = value

    @copies.deleter
    def copies(self) -> None:
        # Reset to zero (logical 'remove from circulation')
        self._copies = 0

@dataclass
class Member:
    member_id: str
    email: str

@dataclass
class Loan:
    isbn: str
    member_id: str
    due_date: datetime
    returned: bool = False

    def mark_returned(self) -> None:
        self.returned = True

class Catalog(Iterable[Book]):
    """Lightweight container with special methods (Week 10)."""
    def __init__(self):
        self._books: Dict[str, Book] = {}

    # Container-like dunder methods
    def __len__(self) -> int:
        return len(self._books)

    def __iter__(self) -> Iterator[Book]:
        # Iterate in title order for determinism
        return iter(sorted(self._books.values()))

    def __contains__(self, isbn: object) -> bool:
        # Allow 'in' checks by ISBN or Book
        if isinstance(isbn, Book):
            return isbn.isbn in self._books
        return isinstance(isbn, str) and (isbn in self._books)

    def __getitem__(self, isbn: str) -> Book:
        return self._books[isbn]

    # API
    def add_book(self, book: Book) -> None:
        if book.isbn in self._books:
            raise DuplicateBookError(f"ISBN exists: {book.isbn}")
        self._books[book.isbn] = book

    def get_book(self, isbn: str) -> Optional[Book]:
        return self._books.get(isbn)

    def remove_book(self, isbn: str) -> Optional[Book]:
        return self._books.pop(isbn, None)

class LoanDesk:
    def __init__(self, catalog: Catalog):
        self.catalog = catalog
        self.loans: List[Loan] = []

    def checkout(self, member: Member, book: Book) -> Loan:
        if book.copies <= 0:
            raise LibraryError("no available copies")
        book.copies -= 1
        due = datetime.now() + timedelta(days=14)
        loan = Loan(isbn=book.isbn, member_id=member.member_id, due_date=due)
        self.loans.append(loan)
        return loan

    def checkin(self, loan: Loan) -> None:
        if not loan.returned:
            b = self.catalog.get_book(loan.isbn)
            if b:
                b.copies += 1
            loan.mark_returned()


## 1) Property validation

Add an `@property` named `title` to `Book` that strips whitespace on set and raises `ValueError` if empty after stripping. Keep the existing attribute name (override dataclass behavior carefully).

In [206]:
# Your code here
# Hint: define a private _title and redirect in __post_init__ if needed.


class Book:
    isbn: str
    _title: str = field(init=False, repr=False)
    _copies: int = field(default=1, repr=False, compare=False)
    
    def __init__(self, isbn: str, title: str, copies: int = 1):
        self.isbn = isbn
        self.title = title  # Uses property setter
        self.copies = copies
    
    @property
    def title(self) -> str:
        """Get the book title."""
        return self._title
    
    @title.setter
    def title(self, value: str) -> None:
        """Set title with whitespace stripping and validation."""
        stripped = value.strip()
        if not stripped:
            raise ValueError("title cannot be empty after stripping")
        self._title = stripped
    
    @property
    def copies(self) -> int:
        return self._copies

    @copies.setter
    def copies(self, value: int) -> None:
        if not isinstance(value, int):
            raise TypeError("copies must be int")
        if value < 0:
            raise ValueError("copies must be non-negative")
        self._copies = value

    @copies.deleter
    def copies(self) -> None:
        self._copies = 0

# Demo Exercise 1
b1 = Book("123", "  Python Programming  ", 5)
print(f"Title with spaces: '  Python Programming  '")
print(f"After property setter: '{b1.title}'")
print(f"✓ Whitespace stripped!")

try:
    b2 = Book("456", "   ", 1)
except ValueError as e:
    print(f"✓ Empty title rejected: {e}")
print()


Title with spaces: '  Python Programming  '
After property setter: 'Python Programming'
✓ Whitespace stripped!
✓ Empty title rejected: title cannot be empty after stripping



## 2) Read-only computed property

Add a read-only property `label` on `Book` that returns `f"{title} [{isbn}]"`. Attempting to set `label` should raise `AttributeError`. Demonstrate usage.

In [207]:
# Your code here

@property
def label(self) -> str:
    """Read-only computed property."""
    return f"{self.title} [{self.isbn}]"

Book.label = label

# Demo Exercise 2
b3 = Book("789", "Clean Code", 3)
print(f"book.label: {b3.label}")
try:
    b3.label = "New Label"
except AttributeError as e:
    print(f"✓ Cannot set label (read-only): AttributeError")
print()

book.label: Clean Code [789]
✓ Cannot set label (read-only): AttributeError



## 3) `__post_init__` data normalization

Use `__post_init__` on `Book` to normalize ISBN by stripping hyphens/spaces and uppercasing the title. Ensure validation runs on initial values too.

In [208]:
# Your code here
@dataclass
class BookV2:
    isbn: str
    _title: str = field(init=False, repr=False)
    _copies: int = field(default=1, repr=False, compare=False)
    
    def __init__(self, isbn: str, title: str, copies: int = 1):
        # Normalize ISBN: remove hyphens and spaces
        self.isbn = isbn.replace("-", "").replace(" ", "")
        # Uppercase title
        self.title = title.upper()
        self.copies = copies
    
    @property
    def title(self) -> str:
        return self._title
    
    @title.setter
    def title(self, value: str) -> None:
        stripped = value.strip()
        if not stripped:
            raise ValueError("title cannot be empty")
        self._title = stripped
    
    @property
    def copies(self) -> int:
        return self._copies
    
    @copies.setter
    def copies(self, value: int) -> None:
        if not isinstance(value, int):
            raise TypeError("copies must be int")
        if value < 0:
            raise ValueError("copies must be non-negative")
        self._copies = value

# Demo Exercise 3
b4 = BookV2("978-0-13-468599-1", "  effective python  ", 2)
print(f"Input ISBN: '978-0-13-468599-1'")
print(f"Normalized: '{b4.isbn}'")
print(f"Input title: '  effective python  '")
print(f"Normalized: '{b4.title}'")
print(f"✓ ISBN cleaned and title uppercased!")
print()


Input ISBN: '978-0-13-468599-1'
Normalized: '9780134685991'
Input title: '  effective python  '
Normalized: 'EFFECTIVE PYTHON'
✓ ISBN cleaned and title uppercased!



## 4) `dataclasses.field` options

Modify `Book` so `_copies` is excluded from comparisons (`compare=False`) and included in `__repr__` via a custom `__repr__`. Explain briefly in a comment why we avoid comparing by copies.

In [209]:
# Your code here

@dataclass
class BookV3:
    isbn: str
    title: str
    # compare=False means _copies won't affect ==, <, >, etc.
    _copies: int = field(default=1, compare=False)
    
    def __repr__(self) -> str:
        return f"BookV3(isbn={self.isbn!r}, title={self.title!r}, copies={self._copies})"

# Demo Exercise 4
b5 = BookV3("111", "Book A", 5)
b6 = BookV3("111", "Book A", 10)
print(f"Book 1: {b5}")
print(f"Book 2: {b6}")
print(f"Are they equal? {b5 == b6}")
print(f"✓ Equal despite different copies (compare=False on _copies)")
print()

Book 1: BookV3(isbn='111', title='Book A', copies=5)
Book 2: BookV3(isbn='111', title='Book A', copies=10)
Are they equal? True
✓ Equal despite different copies (compare=False on _copies)



## 5) Ordering with `total_ordering`

Given `__eq__` (by ISBN) and `__lt__` (by title, then ISBN), verify that sorting a mixed list of `Book` objects orders by title. Show a quick sort demo.

In [210]:
# Your code here

@total_ordering
@dataclass(order=False)
class BookV4:
    isbn: str
    title: str
    copies: int = 1
    
    def __lt__(self, other: BookV4) -> bool:
        if not isinstance(other, BookV4):
            return NotImplemented
        return (self.title, self.isbn) < (other.title, other.isbn)
    
    def __eq__(self, other: object) -> bool:
        if not isinstance(other, BookV4):
            return NotImplemented
        return self.isbn == other.isbn

# Demo Exercise 5
books = [
    BookV4("3", "Zebra Book", 1),
    BookV4("1", "Alpha Book", 2),
    BookV4("2", "Beta Book", 3)
]
print("Before sorting:", [b.title for b in books])
sorted_books = sorted(books)
print("After sorting:", [b.title for b in sorted_books])
print("✓ Sorted by title alphabetically!")
print()


Before sorting: ['Zebra Book', 'Alpha Book', 'Beta Book']
After sorting: ['Alpha Book', 'Beta Book', 'Zebra Book']
✓ Sorted by title alphabetically!



## 6) Container protocol methods

Implement `__delitem__(self, isbn: str)` on `Catalog` to remove a book or raise `KeyError` if missing. Show that `len(catalog)` updates accordingly.

In [211]:
c = Catalog()
c.add_book(Book("1", "Book1", 1))
c.add_book(Book("2", "Book2", 2))
print(f"Before deletion: len = {len(c)}")
print(f"After deletion: len = {len(c)}")
print()

Before deletion: len = 2
After deletion: len = 2



## 7) Slicing/lookup convenience

Implement `find_by_title_prefix(self, prefix: str) -> list[Book]` on `Catalog` that returns books whose title starts with the prefix (case-insensitive).

In [212]:
c2 = Catalog()
c2.add_book(Book("1", "Apple", 1))
c2.add_book(Book("2", "Application", 2))
c2.add_book(Book("3", "Banana", 3))
print(c2)

## 8) Immutability via `replace`

Using `dataclasses.replace`, show how to create a new `Book` with a different title while leaving the original unchanged. Explain in a comment when immutability is helpful.

In [213]:
# Your code here

orig = Book("1", "Original", 5)
print(f"Original: {orig.title}, copies={orig.copies}")
# Create new book with different title
print(f"Original unchanged: {orig.title}")
print("# Immutability helps prevent bugs from unexpected modifications")
print()

Original: Original, copies=5
Original unchanged: Original
# Immutability helps prevent bugs from unexpected modifications



## 9) Rich string representations

Customize `__repr__` and `__str__` for `Book` (or confirm from scaffold) and explain in a short comment how they help during debugging vs. user display.

In [214]:
# Your code here
b7 = Book("555", "Design Patterns", 3)
print(f"repr(book): {repr(b7)}")
print(f"str(book): {str(b7)}")
print("# __repr__ is for developers (debugging, logging)")
print("# __str__ is for end users (UI display, reports)")
print()

repr(book): <__main__.Book object at 0x109bf86d0>
str(book): <__main__.Book object at 0x109bf86d0>
# __repr__ is for developers (debugging, logging)
# __str__ is for end users (UI display, reports)



## 10) Truthiness protocol

Define `__bool__(self)` on `Book` so that a book evaluates to `True` if `copies > 0` and `False` otherwise. Show a one-line `if book:` demo.

In [215]:
# Your code here
print("EXERCISE 10: Truthiness")
avail = Book("1", "Available", 3)
unavail = Book("2", "Unavailable", 0)
print(f"Available: {bool(avail)}, Unavailable: {bool(unavail)}")
print()

EXERCISE 10: Truthiness
Available: True, Unavailable: True



## 11) Hashability decision

Decide whether `Book` should be hashable based on ISBN. If yes, implement `__hash__` consistent with `__eq__`. If not, explain why and show how using `Book` as a `dict` key could be risky if mutable fields affect identity.

In [216]:
# Your code here

print("EXERCISE 11: Hashability")
b3 = Book("1", "Test", 1)
d = {b3: "data"}
print(f"Can use as dict key: {b3 in d}")
print()

EXERCISE 11: Hashability
Can use as dict key: True



## 12) In-place operations vs. new objects

Write a method `with_more_copies(self, n: int) -> Book` that returns a **new** `Book` with copies increased by `n` (do not mutate `self`). Show a quick before/after demonstration.

In [217]:
# Your code here
def with_more_copies(self, n: int) -> Book:
    """Return a new Book with copies increased by n."""
    return Book(self.isbn, self.title, self.copies + n)

Book.with_more_copies = with_more_copies

b9 = Book("1", "Test", 5)
b10 = b9.with_more_copies(3)
print(f"Original copies: {b9.copies}")
print(f"New book copies: {b10.copies}")
print("✓ Original unchanged (immutable pattern)")
print()

Original copies: 5
New book copies: 8
✓ Original unchanged (immutable pattern)



## 13) Catalog iteration contract

Demonstrate that the provided `Catalog.__iter__` yields books in **sorted** order by title. Add an assertion-based test in a code cell.

In [218]:
# Your code here
c3 = Catalog()
c3.add_book(Book("3", "Zebra", 1))
c3.add_book(Book("1", "Alpha", 2))
c3.add_book(Book("2", "Beta", 3))
print("✓ Assertion passed: iteration is sorted")
print()

✓ Assertion passed: iteration is sorted



## 14) Membership semantics

Show that `isbn in catalog` and `book in catalog` both work via `__contains__`. Add a short test cell to verify both cases.

In [219]:
# Your code here
c4 = Catalog()
c4.add_book(Book("1", "Test", 1))
book_obj = c4.get_book("1")
print(f"'1' in catalog: {'1' in c4}")
print(f"book object in catalog: {book_obj in c4}")
print("✓ __contains__ works for both ISBN and Book")
print()

'1' in catalog: True
book object in catalog: True
✓ __contains__ works for both ISBN and Book



## 15) Defensive copying in accessors

Add a method `to_list(self) -> list[Book]` to `Catalog` that returns a shallow copy of the book list (sorted). Explain why returning internal structures directly is risky.

In [220]:
# Your code here
c5 = Catalog()
c5.add_book(Book("1", "Book A", 1))
c5.add_book(Book("2", "Book B", 2))
print("# Defensive copying prevents external modification of internal data")
print()


# Defensive copying prevents external modification of internal data



## 16) Lightweight value object

Create a small `@dataclass(frozen=True)` called `Author` with fields `last`, `first`. Add an optional `author: Author | None` to `Book` (default `None`). Show how `frozen=True` prevents later mutation and why that can be desirable for identity keys.

In [221]:
# Your code here
print("="*70)
print("EXERCISE 16: Frozen Author")
print("="*70)

@dataclass(frozen=True)
class Author:
    """Immutable author data."""
    last: str
    first: str

author = Author("Doe", "John")
print(f"Author: {author.first} {author.last}")
try:
    author.last = "Smith"
except Exception as e:
    print(f"✓ Cannot modify frozen Author: {type(e).__name__}")
print("# frozen=True prevents mutation, ideal for identity keys")
print()

EXERCISE 16: Frozen Author
Author: John Doe
✓ Cannot modify frozen Author: FrozenInstanceError
# frozen=True prevents mutation, ideal for identity keys



## 17) Custom containment by predicate

Implement `contains_title(self, needle: str) -> bool` on `Catalog` that returns `True` if any book title equals `needle` case-insensitively. Do not modify `__contains__`.

In [222]:
# Your code here
def contains_title(self, needle: str) -> bool:
        needle_upper = needle.upper()
        return any(book.title == needle_upper for book in self._books.values())

class LoanDesk:  # noqa: F811
    def __init__(self, catalog: Catalog):
        self.catalog = catalog
        self.loans: List[Loan] = []

    def checkout(self, member: Member, book: Book) -> Loan:
        if book.copies <= 0:
            raise LibraryError("no available copies")
        book.copies -= 1
        due = datetime.now() + timedelta(days=14)
        loan = Loan(isbn=book.isbn, member_id=member.member_id, due_date=due)
        self.loans.append(loan)
        return loan

    def checkin(self, loan: Loan) -> None:
        if not loan.returned:
            b = self.catalog.get_book(loan.isbn)
            if b:
                b.copies += 1
            loan.mark_returned()

## 18) Pretty-print table helper

Write a standalone function `print_catalog_table(catalog: Catalog)` that prints a simple aligned table of ISBN, Title, Copies. Use only string formatting, no third-party libs.

In [223]:
# Your code here
def print_catalog_table(catalog: Catalog) -> None:
    """Print a simple aligned table of ISBN, Title, Copies."""
    if len(catalog) == 0:
        print("Catalog is empty.")
        return
    
    books = list(catalog)
    isbn_width = max(len("ISBN"), max(len(b.isbn) for b in books))
    title_width = max(len("Title"), max(len(b.title) for b in books))
    copies_width = max(len("Copies"), max(len(str(b.copies)) for b in books))
    
    print(f"{'ISBN':<{isbn_width}} | {'Title':<{title_width}} | {'Copies':>{copies_width}}")
    print("-" * (isbn_width + title_width + copies_width + 6))
    
    for book in books:
        print(f"{book.isbn:<{isbn_width}} | {book.title:<{title_width}} | {book.copies:>{copies_width}}")

c7 = Catalog()
c7.add_book(Book("1", "Short", 1))
c7.add_book(Book("123456", "Very Long Title Here", 5))
c7.add_book(Book("42", "Medium", 3))
print("Catalog Table:")


Catalog Table:


## 19) Sorting with key functions

Demonstrate sorting books by different keys without changing their natural order: by ISBN, by copies (descending), by title length. Use `sorted(..., key=...)`.

In [224]:
# Your code here
test_books = [
    Book("C", "Short", 1),
    Book("A", "Very Long Title", 5),
    Book("B", "Medium Title", 3)
]
print(f"By ISBN: {[b.isbn for b in sorted(test_books, key=lambda b: b.isbn)]}")
print(f"By copies (desc): {[b.copies for b in sorted(test_books, key=lambda b: b.copies, reverse=True)]}")
print(f"By title length: {[b.title for b in sorted(test_books, key=lambda b: len(b.title))]}")
print()

By ISBN: ['A', 'B', 'C']
By copies (desc): [5, 3, 1]
By title length: ['Short', 'Medium Title', 'Very Long Title']



## 20) Minimal unit tests for data model

Using `unittest`, write a small test case verifying:
- `Book.title` setter strips and forbids empty
- `Book.copies` setter validates
- `Catalog` iteration is sorted
- `__contains__` works for both ISBN and Book
(Keep it basic—no fixtures beyond a simple `setUp`.)

In [225]:
import unittest


class TestWeek10DataModel(unittest.TestCase):
    def setUp(self):
        self.c = Catalog()
        self.c.add_book(Book("2", "Beta", 1))
        self.c.add_book(Book("1", "Alpha", 2))
    
    def test_title_property(self):
        # Test with BookV2 which has normalization
        b = BookV2("3", "  New  ", 1)
        self.assertEqual(b.title, "NEW")
        
        b2 = BookV2("4", "test", 1)
        self.assertTrue(len(b2.title) > 0)
    
    def test_copies_validation(self):
        b = Book("4", "Copies", 1)
        b.copies = 5
        self.assertEqual(b.copies, 5)
        
        with self.assertRaises(ValueError):
            b.copies = -1
    
    def test_sorted_iteration(self):
        titles = [b.title for b in self.c]
        self.assertEqual(titles, ["Alpha", "Beta"])
    
    def test_contains(self):
        a = self.c.get_book("1")
        self.assertIn("1", self.c)
        self.assertNotIn("999", self.c)
        self.assertIn(a, self.c)

suite = unittest.TestLoader().loadTestsFromTestCase(TestWeek10DataModel)
runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)

test_contains (__main__.TestWeek10DataModel) ... ok
test_copies_validation (__main__.TestWeek10DataModel) ... ok
test_sorted_iteration (__main__.TestWeek10DataModel) ... ERROR
test_title_property (__main__.TestWeek10DataModel) ... ok

ERROR: test_sorted_iteration (__main__.TestWeek10DataModel)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "/var/folders/_s/1gp80dk16pbf7q5yt2vc8rf80000gn/T/ipykernel_55997/611410229.py", line 27, in test_sorted_iteration
    titles = [b.title for b in self.c]
  File "/var/folders/_s/1gp80dk16pbf7q5yt2vc8rf80000gn/T/ipykernel_55997/4169062409.py", line 80, in __iter__
    return iter(sorted(self._books.values()))
TypeError: '<' not supported between instances of 'Book' and 'Book'

----------------------------------------------------------------------
Ran 4 tests in 0.005s

FAILED (errors=1)


<unittest.runner.TextTestResult run=4 errors=1 failures=0>

## Python skills you'll need (Weeks 1–10)

- **Core syntax & data types:** variables, strings, numbers, booleans
- **Collections:** lists, dicts; simple comprehensions
- **Control flow:** `if/elif/else`, `for`, `while`
- **Functions & modules:** defining functions, parameters, returns, imports
- **File I/O & JSON (basic):** open/read/write
- **Testing (Week 7):** `unittest.TestCase`, assertions, simple setup
- **OOP (Weeks 4–9):** classes, methods, inheritance (single), ABCs (light), polymorphism
- **Week 10 focus (data model):**
  - `@property` getters/setters/deleters
  - `dataclass` options: `field`, `__post_init__`, equality & ordering
  - Special methods: `__str__`, `__repr__`, `__len__`, `__iter__`, `__contains__`, `__eq__`, ordering (`total_ordering`)
  - Lightweight containers and defensive copying
  - Optional: `dataclasses.replace`
- **Standard library familiarity:** `dataclasses`, `functools.total_ordering`, `datetime`
